<a href="https://colab.research.google.com/drive/1SPG3mGHRVME1TrXM3Y1As6y_Tj3VjQSX?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![CuratorKIT](CuratorKIT_logo-TP.png)

---

# Data ingestion & cleaning

**Pattern:** 3 datasets, each with its own `preprocessing_fn` and `max_samples` cap, merged and resampled.

Ingests heterogeneous datasets (Alpaca, hh-rlhf, GSM8K), normalizes each to a uniform
`instruction_following` schema via custom preprocessing functions, then merges, deduplicates,
cleans, and resamples by source dataset to control each source's contribution.

### Pipeline flow
```
yahma/alpaca-cleaned (5k) ──► preprocess_alpaca()
Anthropic/hh-rlhf (3k)    ──► preprocess_hh_rlhf()   ──► SchemaGate → ExactDedup → MinHashDedup
gsm8k/main (2k)           ──► preprocess_gsm8k()          → TextCleaner → DiversityGate
                                                           → EmbeddingDedup (cross-run)
                                                           → StratifiedSampler (50/30/20)
                                                           → AlpacaExporter (train/val/test split)
```

In [2]:
# ═══════════════════════════════════════════════════════════════════════════
# 0. Setup — clone + install CuratorKIT (run once, then comment out)
# Prerequisite: SSH key added to your GitHub account with repo access.
# ═══════════════════════════════════════════════════════════════════════════
from pathlib import Path

repo_dir = Path.home() / "CuratorKIT"
if not repo_dir.exists():
    !git clone https://github.com/Lexsi-Labs/CuratorKIT.git {repo_dir}

!pip install -e "{repo_dir}[generation,embedding,hf,parquet]" -q

print(f"CuratorKIT installed from {repo_dir}")

# !pip install "/content/curatorkit.tar.gz[generation,embedding,hf,parquet,pdf-gpu]" nest-asyncio -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 10.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.6/786.6 kB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

## 1. Preprocessing functions

Each dataset has its own schema. These functions normalize them to `instruction/input/output`
and tag a `source_dataset` key. Unmapped keys in the returned dict are automatically
forwarded to `DataSample.metadata` by the connector, where `StratifiedSampler` reads them.

In [3]:
def preprocess_alpaca(row: dict) -> dict:
    """yahma/alpaca-cleaned — already standard, just tag source."""
    return {
        "instruction":    row.get("instruction", ""),
        "input":          row.get("input", ""),
        "output":         row.get("output", ""),
        "task_type":      "instruction_following",
        "source_dataset": "alpaca",    # forwarded to metadata by the connector
    }


def preprocess_hh_rlhf(row: dict) -> dict:
    """
    Anthropic/hh-rlhf — full dialogue in 'chosen' with Human:/Assistant: markers.
    Extract the last Human turn as instruction and final Assistant turn as output.
    """
    chosen = row.get("chosen", "")
    segments = chosen.strip().split("\n\nHuman: ")
    last = segments[-1] if segments else chosen
    if "\n\nAssistant: " in last:
        instruction, output = last.split("\n\nAssistant: ", 1)
    else:
        instruction, output = last, ""
    return {
        "instruction":    instruction.strip(),
        "input":          "",
        "output":         output.strip(),
        "task_type":      "instruction_following",
        "source_dataset": "hh-rlhf",   # forwarded to metadata by the connector
    }


def preprocess_gsm8k(row: dict) -> dict:
    """gsm8k — math word problems: question → instruction, answer → output."""
    return {
        "instruction":    row.get("question", ""),
        "input":          "",
        "output":         row.get("answer", ""),
        "task_type":      "instruction_following",
        "source_dataset": "gsm8k",      # forwarded to metadata by the connector
    }

## 2. Run pipeline

In [4]:
from curatorkit import Curator, CuratorConfig

config = CuratorConfig(
    # ── 3 sources, each with its own cap and preprocessing function ────────
    # Per-reader max_samples caps each dataset independently before merging.
    # Without this, alpaca's 52k rows would fill the entire pipeline.
    dataset=[
        {"name": "yahma/alpaca-cleaned", "max_samples": 1500},
        {"name": "Anthropic/hh-rlhf",    "split": "train", "max_samples": 1000},
        {"name": "gsm8k", "subset": "main", "split": "train", "max_samples": 1000},
    ],
    preprocessing_fn=[
        preprocess_alpaca,
        preprocess_hh_rlhf,
        preprocess_gsm8k,
    ],
    split="train",

    # ── Schema gate ───────────────────────────────────────────────────────
    min_tokens=10,
    max_tokens=1024,

    # ── Dedup ─────────────────────────────────────────────────────────────
    dedup="minhash",
    minhash_threshold=0.85,

    # ── Cross-run embedding dedup ─────────────────────────────────────────
    embedding_dedup=True,
    embedding_dedup_threshold=0.90,
    embedding_index_dir="output/corpus_index",
    embedding_reset_index=True,
    embedding_device="cuda",

    # ── Text cleaning ─────────────────────────────────────────────────────
    clean=True,

    # ── Diversity gate ────────────────────────────────────────────────────
    diversity_threshold=0.92,

    # ── Resample by source_dataset ────────────────────────────────────────
    # All samples are instruction_following after normalization, so resampling
    # by task_type would be meaningless. Resample by source_dataset instead.
    resample=True,
    resample_field="source_dataset",
    target_distribution={"alpaca": 0.5, "hh-rlhf": 0.3, "gsm8k": 0.2},

    # ── Export with train/val/test split ──────────────────────────────────
    export_formats=["alpaca"],
    output_dir="output/data_ingestion_multi_reader",
    output_split={"train": 0.8, "val": 0.1, "test": 0.1},
)

curator = Curator(config)
curator.dry_run()

print("\n=== Running ===")
result = curator.run()
result.print_summary()


=== Pipeline.dry_run — 11 step(s) ===
Output dir: output/data_ingestion_multi_reader

   1. [reader    ] HuggingFaceReader
   2. [reader    ] HuggingFaceReader
   3. [reader    ] HuggingFaceReader
   4. [gate      ] SchemaGate  (cfg=ffff512734781804)
   5. [normalizer] ExactDeduplicator  (cfg=98ec237522ba1e05)
   6. [normalizer] MinHashDeduplicator  (cfg=9315210001ce73f7)
   7. [normalizer] TextCleaner  (cfg=2176e3c3449cf50b)
   8. [gate      ] DiversityGate  (cfg=7ec497e4b545234e)
   9. [normalizer] EmbeddingDeduplicator  (cfg=069851cea196c05f)
  10. [normalizer] StratifiedSampler  (cfg=f2c307680ede9f0c)
  11. [exporter  ] AlpacaExporter

Always emitted: manifest.json, dataset_card.md, rejected.jsonl, checksums.txt
=== END Pipeline.dry_run ===


=== Running ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

StratifiedSampler: 100%|██████████| 3427/3427 [00:00<00:00, 153907.46sample/s]


────────────────────────────────────────────
  passed   :    2,930
  rejected :       65
  time     :   132.4s
  output   : output/data_ingestion_multi_reader
────────────────────────────────────────────


## 3. Inspect

In [5]:
from collections import Counter

# ── Source distribution after resampling ──────────────────────────────────
sources = Counter(s.metadata.get("source_dataset", "?") for s in result.passed)
print("Source distribution in final output:")
for src, count in sources.most_common():
    pct = 100 * count / len(result.passed)
    print(f"  {src:<12} {count:>5}  ({pct:.0f}%)")

print(f"\nRejected: {len(result.rejected):,}")
if result.rejected:
    reasons = Counter(r.rejection_reason for r in result.rejected)
    for reason, count in reasons.most_common(5):
        print(f"  {count:>4}  {reason}")

# ── Sample preview ────────────────────────────────────────────────────────
print("\nSample:")
s = result.passed[0]
print(f"  source : {s.metadata.get('source_dataset', '?')}")
print(f"  Q: {s.instruction[:]}")
print(f"  A: {s.output[:100]}")

Source distribution in final output:
  alpaca        1465  (50%)
  hh-rlhf        879  (30%)
  gsm8k          586  (20%)

Rejected: 65
    20  below_min_tokens:9
    13  below_min_tokens:8
     9  below_min_tokens:7
     5  below_min_tokens:5
     4  below_min_tokens:6

Sample:
  source : alpaca
  Q: Complete a sentence that means the same as the following sentence, but is more concise:
  A: He was clueless.


In [5]:
# ── Output files ──────────────────────────────────────────────────────────
from pathlib import Path
out = Path("output/data_ingestion_multi_reader")
print(f"{out}/")
for p in sorted(out.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(out)}")

output/data_ingestion_multi_reader/
  checksums.txt
  dataset_card.md
  manifest.json
  rejected.jsonl
  test/sft_alpaca.jsonl
  train/sft_alpaca.jsonl
  val/sft_alpaca.jsonl


## 4. Key design decisions

**Why per-reader max_samples?**
Each dataset gets its own cap applied during `read()`, before samples are concatenated.
A global `max_samples` would fill entirely from the first reader (alpaca has 52k rows).
Per-reader caps ensure proportional contribution.

**Why resample by source_dataset?**
After preprocessing, all samples have `task_type="instruction_following"`.
Resampling by `source_dataset` controls each dataset's weight in the final pool.
The `source_dataset` key is set in the preprocessing dict and forwarded to
`DataSample.metadata` by the connector's `_collect_metadata` method.

**Why EmbeddingDedup with reset_index=True?**
First run starts fresh. Subsequent runs set `embedding_reset_index=False` for
incremental cross-run dedup — new samples checked against all previous runs.